# **3. Train and Test ML algorithm in Materials - Unsupervised Learning(Assignment)**

In [3]:
!pip install matminer[citrine] -q # install matminer library
!pip install pyyaml -q
!pip install pymatgen -q
!pip install --upgrade pyyaml six matminer[citrine] citrination-client pymatgen -q
!pip install numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.1/809.1 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.0 MB/s eta 0:00:00
   ━━━━

In [4]:
# Task: Load the elastic tensor dataset from matminer and keep only the columns
# "material_id", "K_VRH", "G_VRH", and "poisson_ratio"
from matminer.datasets.convenience_loaders import load_elastic_tensor

df = load_elastic_tensor()  # load the dataset in a pandas DataFrame object
print(df.columns)  # check columns
df = df[["material_id", "K_VRH", "G_VRH", "poisson_ratio"]]
df.head()


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# Task: Exclude the column "material_id" from the dataset and create a new DataFrame X
# containing only the numerical feature columns
excluded = ["material_id"]
X = df.drop(excluded, axis=1)
X.head()


In [ ]:
# Task: Perform unstructured Agglomerative Clustering (Ward linkage) on dataset X with 6 clusters,
# measure the elapsed time, and obtain the cluster labels

#The agglomerative clustering is the most common type of hierarchical clustering used to group objects in clusters based on their similarity.
#It’s also known as AGNES (Agglomerative Nesting). The algorithm starts by treating each object as a singleton cluster.
#Next, pairs of clusters are successively merged until all clusters have been merged into one big cluster containing all objects.
import time as time
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import AgglomerativeClustering
import sklearn.datasets
#The method of merging two clusters involves combining the two clusters that result in the smallest increase in the variance within all clusters.(Ward)
print("Compute unstructured hierarchical clustering...")
st = time.time()
ward = AgglomerativeClustering(n_clusters=6, linkage="ward").fit(X)
elapsed_time = time.time() - st
group = ward.labels_
print(f"Elapsed time: {elapsed_time:.2f}s")
print(f"Number of points: {group.size}")


In [ ]:
group

In [ ]:
# Task: Add the cluster labels as a new column "group" to the DataFrame and display the updated DataFrame
df['group'] = group.tolist()
df


In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from mpl_toolkits.mplot3d import Axes3D  # for 3D plotting

X_3D = df[["K_VRH", "G_VRH", "poisson_ratio"]].dropna()

fig = plt.figure(figsize=(15, 10))

for i, n_clusters in enumerate(range(1, 7), 1):  # 1 ~ 6 clusters
    ax = fig.add_subplot(2, 3, i, projection="3d")

    model = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
    labels = model.fit_predict(X_3D)

    ax.scatter(X_3D["K_VRH"], X_3D["G_VRH"], X_3D["poisson_ratio"],
               c=labels, cmap="tab10", s=5)

    ax.set_title(f"n_clusters = {n_clusters}")
    ax.set_xlabel("K_VRH")
    ax.set_ylabel("G_VRH")
    ax.set_zlabel("poisson_ratio")

plt.tight_layout()
plt.show()


In [ ]:
# Task: Perform structured Agglomerative Clustering (Ward linkage) on dataset X with 5 clusters,
# using a k-nearest neighbors connectivity graph (n_neighbors = 5) to impose local connectivity constraints.
# Measure the elapsed time and obtain the cluster labels.

from sklearn.neighbors import kneighbors_graph

connectivity = kneighbors_graph(X, n_neighbors=5, include_self=False)
print("Compute structured hierarchical clustering...")
st = time.time()
ward = AgglomerativeClustering(
    n_clusters=5, connectivity=connectivity, linkage="ward"
).fit(X)
elapsed_time = time.time() - st
group = ward.labels_
print(f"Elapsed time: {elapsed_time:.2f}s")
print(f"Number of points: {group.size}")


In [ ]:
group

In [ ]:
# Task: Add the cluster labels as a new column "group" to the DataFrame and display the updated DataFrame
df['group'] = group.tolist()
df


In [5]:
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from mpl_toolkits.mplot3d import Axes3D  # for 3D plotting

X_3D = df[["K_VRH", "G_VRH", "poisson_ratio"]].dropna()

fig = plt.figure(figsize=(15, 10))

for i, n_clusters in enumerate(range(1, 7), 1):  # 1 ~ 6 clusters
    ax = fig.add_subplot(2, 3, i, projection="3d")
    connectivity = kneighbors_graph(X, n_neighbors=5, include_self=False)
    model = AgglomerativeClustering(n_clusters=n_clusters, connectivity = connectity, linkage="ward")
    labels = model.fit_predict(X_3D)

    ax.scatter(X_3D["K_VRH"], X_3D["G_VRH"], X_3D["poisson_ratio"],
               c=labels, cmap="tab10", s=5)

    ax.set_title(f"n_clusters = {n_clusters}")
    ax.set_xlabel("K_VRH")
    ax.set_ylabel("G_VRH")
    ax.set_zlabel("poisson_ratio")

plt.tight_layout()
plt.show()



ModuleNotFoundError: No module named 'numpy.strings'